In [131]:
import pandas as pd
import numpy as np
import time
from nba_api.stats.endpoints import playergamelog, leaguedashplayerstats

# Reload saved dataset — all features already computed and saved
df_master = pd.read_csv('nba_master_dataset.csv', parse_dates=['GAME_DATE'])

print(f"df_master loaded: {len(df_master):,} rows")
print(f"Columns: {len(df_master.columns)}")
print()

# Verify key features are present
key_features = [
    'DAYS_REST', 'DEF_RATING', 'PACE',
    'OPP_PTS_ALLOWED_PG', 'USG_PCT',
    'OPP_PTS_VS_POS', 'OPP_PTS_VS_POS_roll5',
    'USG_PCT_roll5'
]
print("Key feature check:")
for col in key_features:
    status = "✅" if col in df_master.columns else "❌ MISSING"
    print(f"  {col}: {status}")

df_master loaded: 54,701 rows
Columns: 76

Key feature check:
  DAYS_REST: ✅
  DEF_RATING: ✅
  PACE: ✅
  OPP_PTS_ALLOWED_PG: ✅
  USG_PCT: ✅
  OPP_PTS_VS_POS: ✅
  OPP_PTS_VS_POS_roll5: ✅
  USG_PCT_roll5: ✅


In [133]:
# Season stats and 20+ min filter

from nba_api.stats.endpoints import leaguedashplayerstats
import time

# Pull season averages for all players in 2024-25
season_stats = leaguedashplayerstats.LeagueDashPlayerStats(
    season='2024-25',
    season_type_all_star='Regular Season'
)
time.sleep(1)

df_season = season_stats.get_data_frames()[0]

# Calculate per game minutes
df_season['MIN_PG'] = df_season['MIN'] / df_season['GP']
df_season['PTS_PG'] = df_season['PTS'] / df_season['GP']
df_season['REB_PG'] = df_season['REB'] / df_season['GP']
df_season['AST_PG'] = df_season['AST'] / df_season['GP']

# Filter to 20+ min players
df_qualified = df_season[df_season['MIN_PG'] >= 20].copy()

print(f"Players averaging 20+ min per game: {len(df_qualified)}")
print()
print(df_qualified[['PLAYER_NAME', 'MIN_PG', 'PTS_PG', 'REB_PG', 'AST_PG']]
      .sort_values('MIN_PG', ascending=False)
      .round(1)
      .head(20))

Players averaging 20+ min per game: 271

         PLAYER_NAME  MIN_PG  PTS_PG  REB_PG  AST_PG
545     Tyrese Maxey    37.7    26.3     3.3     6.1
298        Josh Hart    37.6    13.6     9.6     5.9
141     Devin Booker    37.3    25.6     4.1     7.1
398    Mikal Bridges    37.0    17.6     3.2     3.7
423     Nikola Jokić    36.7    29.6    12.7    10.2
428       OG Anunoby    36.6    18.0     4.8     2.2
330     Kevin Durant    36.5    26.6     6.0     4.2
263     Jayson Tatum    36.4    26.8     8.7     6.0
29   Anthony Edwards    36.3    27.6     5.7     4.5
126     De'Aaron Fox    36.1    23.5     4.8     6.3
238     Jamal Murray    36.1    21.4     3.9     6.0
112   Damian Lillard    36.1    24.9     4.7     7.1
351     Kyrie Irving    36.1    24.7     4.8     4.6
523       Trae Young    36.0    24.2     3.1    11.6
130    DeMar DeRozan    35.9    22.2     3.9     4.4
230    Jalen Johnson    35.7    18.9    10.0     5.0
226    Jalen Brunson    35.4    26.0     2.9     7.3
540  

In [135]:
# Define clean player log function

def clean_player_log(df, player_name, season):
    """
    Takes a raw game log dataframe and returns a clean version
    with rolling averages.
    """
    # Add player name and season columns
    df['PLAYER_NAME'] = player_name
    df['SEASON']      = season

    # Fix dates
    df['GAME_DATE'] = pd.to_datetime(df['GAME_DATE'], format='mixed')

    # Parse home/away
    df['HOME_AWAY'] = df['MATCHUP'].apply(
        lambda x: 'HOME' if 'vs.' in x else 'AWAY'
    )

    # Filter low minutes
    df = df[df['MIN'] >= 15].copy()

    # Sort oldest to newest
    df = df.sort_values('GAME_DATE').reset_index(drop=True)

    # Build rolling averages
    target_stats       = ['PTS', 'REB', 'AST', 'BLK', 'STL', 'FG3M']
    feature_only_stats = ['MIN', 'TOV', 'FGA', 'FG3A']
    all_stats          = target_stats + feature_only_stats

    # 5-game rolling average
    for stat in all_stats:
        df[f'{stat}_roll5'] = df[stat].rolling(window=5).mean().shift(1)

    # 10-game rolling average
    for stat in all_stats:
        df[f'{stat}_roll10'] = df[stat].rolling(window=10).mean().shift(1)

    # Drop NaN rows
    df = df.dropna(subset=['PTS_roll5']).reset_index(drop=True)
    return df

print("✅ clean_player_log defined")

✅ clean_player_log defined


In [139]:
# Pull regular season game logs

from nba_api.stats.endpoints import playergamelog
import time

SEASONS = ['2020-21', '2021-22', '2022-23', '2023-24', '2024-25']
SLEEP   = 1.5

# Get qualified player list
player_pull_list = df_qualified[['PLAYER_ID', 'PLAYER_NAME']].reset_index(drop=True)

# Master dataframe — all players, all seasons
all_data = []
total    = len(player_pull_list) * len(SEASONS)
current  = 0

for _, player in player_pull_list.iterrows():
    for season in SEASONS:
        current += 1
        try:
            log = playergamelog.PlayerGameLog(
                player_id=player['PLAYER_ID'],
                season=season
            )
            time.sleep(SLEEP)

            df_raw = log.get_data_frames()[0]

            if len(df_raw) == 0:
                continue

            df_cleaned = clean_player_log(df_raw, player['PLAYER_NAME'], season)
            all_data.append(df_cleaned)

            print(f"[{current}/{total}] ✅ {player['PLAYER_NAME']} {season} — {len(df_cleaned)} rows")

        except Exception as e:
            print(f"[{current}/{total}] ❌ {player['PLAYER_NAME']} {season} — Error: {e}")
            time.sleep(SLEEP)
            continue

# Combine into master dataframe
df_master = pd.concat(all_data, ignore_index=True)

print()
print(f"═══════════════════════════════════")
print(f"  Total rows:    {len(df_master):,}")
print(f"  Total players: {df_master['PLAYER_NAME'].nunique()}")
print(f"  Seasons:       {sorted(df_master['SEASON'].unique())}")
print(f"═══════════════════════════════════")

[3/1355] ✅ AJ Green 2022-23 — 6 rows
[4/1355] ✅ AJ Green 2023-24 — 13 rows
[5/1355] ✅ AJ Green 2024-25 — 60 rows
[10/1355] ✅ AJ Johnson 2024-25 — 14 rows
[11/1355] ✅ Aaron Gordon 2020-21 — 43 rows
[12/1355] ✅ Aaron Gordon 2021-22 — 70 rows
[13/1355] ✅ Aaron Gordon 2022-23 — 62 rows
[14/1355] ✅ Aaron Gordon 2023-24 — 68 rows
[15/1355] ✅ Aaron Gordon 2024-25 — 44 rows
[16/1355] ✅ Aaron Nesmith 2020-21 — 18 rows
[17/1355] ✅ Aaron Nesmith 2021-22 — 8 rows
[18/1355] ✅ Aaron Nesmith 2022-23 — 65 rows
[19/1355] ✅ Aaron Nesmith 2023-24 — 67 rows
[20/1355] ✅ Aaron Nesmith 2024-25 — 35 rows
[22/1355] ✅ Aaron Wiggins 2021-22 — 37 rows
[23/1355] ✅ Aaron Wiggins 2022-23 — 39 rows
[24/1355] ✅ Aaron Wiggins 2023-24 — 38 rows
[25/1355] ✅ Aaron Wiggins 2024-25 — 62 rows
[26/1355] ✅ Al Horford 2020-21 — 23 rows
[27/1355] ✅ Al Horford 2021-22 — 64 rows
[28/1355] ✅ Al Horford 2022-23 — 58 rows
[29/1355] ✅ Al Horford 2023-24 — 59 rows
[30/1355] ✅ Al Horford 2024-25 — 54 rows
[35/1355] ✅ Alex Sarr 2024-25 —

In [69]:
# Adding more features to enhance model performance

# Calculate days rest
import pandas as pd

# Sort by player and date
df_master = df_master.sort_values(['PLAYER_NAME', 'GAME_DATE']).reset_index(drop=True)

# PROTECTION — drop DAYS_REST if it already exists
if 'DAYS_REST' in df_master.columns:
    df_master = df_master.drop(columns=['DAYS_REST'])

# Calculate days between games per player
df_master['DAYS_REST'] = (
    df_master.groupby('PLAYER_NAME')['GAME_DATE']
    .diff()
    .dt.days
)

# Fill first game of each player with league average
df_master['DAYS_REST'] = df_master['DAYS_REST'].fillna(2)

# Cap at 7
df_master['DAYS_REST'] = df_master['DAYS_REST'].clip(upper=7)

print(f"NaNs in DAYS_REST: {df_master['DAYS_REST'].isna().sum()}")
print(df_master[['PLAYER_NAME', 'GAME_DATE', 'DAYS_REST']].head(10))

NaNs in DAYS_REST: 0
  PLAYER_NAME  GAME_DATE  DAYS_REST
0    AJ Green 2023-01-14        2.0
1    AJ Green 2023-02-16        7.0
2    AJ Green 2023-02-24        7.0
3    AJ Green 2023-03-01        5.0
4    AJ Green 2023-03-29        7.0
5    AJ Green 2023-04-07        7.0
6    AJ Green 2024-02-04        7.0
7    AJ Green 2024-02-08        4.0
8    AJ Green 2024-02-09        1.0
9    AJ Green 2024-02-12        3.0


In [141]:
# Add days rest feature

# Sort by player and date
df_master = df_master.sort_values(['PLAYER_NAME', 'GAME_DATE']).reset_index(drop=True)

# Protection
if 'DAYS_REST' in df_master.columns:
    df_master = df_master.drop(columns=['DAYS_REST'])

# Calculate days between games per player
df_master['DAYS_REST'] = (
    df_master.groupby('PLAYER_NAME')['GAME_DATE']
    .diff()
    .dt.days
)

# Fill first game of each player with league average
df_master['DAYS_REST'] = df_master['DAYS_REST'].fillna(2)

# Cap at 7
df_master['DAYS_REST'] = df_master['DAYS_REST'].clip(upper=7)

print(f"NaNs in DAYS_REST: {df_master['DAYS_REST'].isna().sum()}")
print(df_master[['PLAYER_NAME', 'GAME_DATE', 'DAYS_REST']].head(10))

NaNs in DAYS_REST: 0
  PLAYER_NAME  GAME_DATE  DAYS_REST
0    AJ Green 2023-01-14        2.0
1    AJ Green 2023-02-16        7.0
2    AJ Green 2023-02-24        7.0
3    AJ Green 2023-03-01        5.0
4    AJ Green 2023-03-29        7.0
5    AJ Green 2023-04-07        7.0
6    AJ Green 2024-02-04        7.0
7    AJ Green 2024-02-08        4.0
8    AJ Green 2024-02-09        1.0
9    AJ Green 2024-02-12        3.0


In [143]:
# Add opponenet defensive rating

from nba_api.stats.endpoints import leaguedashteamstats
import time

# Pull team defensive stats for each season
opp_def_frames = []

for season in ['2020-21', '2021-22', '2022-23', '2023-24', '2024-25']:
    stats = leaguedashteamstats.LeagueDashTeamStats(
        season=season,
        season_type_all_star='Regular Season',
        measure_type_detailed_defense='Defense'
    )
    time.sleep(1.5)
    df_def = stats.get_data_frames()[0]
    df_def['SEASON'] = season
    opp_def_frames.append(df_def)

df_opp_def = pd.concat(opp_def_frames, ignore_index=True)
df_opp_def = df_opp_def[['TEAM_NAME', 'SEASON', 'DEF_RATING']].copy()

# Extract opponent code from MATCHUP
df_master['OPPONENT'] = df_master['MATCHUP'].apply(lambda x: x.split()[-1])

# Map 3-letter codes to full team names
team_name_map = {
    'ATL': 'Atlanta Hawks', 'BKN': 'Brooklyn Nets',
    'BOS': 'Boston Celtics', 'CHA': 'Charlotte Hornets',
    'CHI': 'Chicago Bulls', 'CLE': 'Cleveland Cavaliers',
    'DAL': 'Dallas Mavericks', 'DEN': 'Denver Nuggets',
    'DET': 'Detroit Pistons', 'GSW': 'Golden State Warriors',
    'HOU': 'Houston Rockets', 'IND': 'Indiana Pacers',
    'LAC': 'LA Clippers', 'LAL': 'Los Angeles Lakers',
    'MEM': 'Memphis Grizzlies', 'MIA': 'Miami Heat',
    'MIL': 'Milwaukee Bucks', 'MIN': 'Minnesota Timberwolves',
    'NOP': 'New Orleans Pelicans', 'NYK': 'New York Knicks',
    'OKC': 'Oklahoma City Thunder', 'ORL': 'Orlando Magic',
    'PHI': 'Philadelphia 76ers', 'PHX': 'Phoenix Suns',
    'POR': 'Portland Trail Blazers', 'SAC': 'Sacramento Kings',
    'SAS': 'San Antonio Spurs', 'TOR': 'Toronto Raptors',
    'UTA': 'Utah Jazz', 'WAS': 'Washington Wizards',
}

df_master['OPP_TEAM_NAME'] = df_master['OPPONENT'].map(team_name_map)

# Protection
if 'DEF_RATING' in df_master.columns:
    df_master = df_master.drop(columns=['DEF_RATING'])

# Merge
df_master = df_master.merge(
    df_opp_def[['TEAM_NAME', 'SEASON', 'DEF_RATING']],
    left_on=['OPP_TEAM_NAME', 'SEASON'],
    right_on=['TEAM_NAME', 'SEASON'],
    how='left'
)
df_master = df_master.drop(columns=['TEAM_NAME'])

print(f"NaNs in DEF_RATING: {df_master['DEF_RATING'].isna().sum()}")
print(df_master[['PLAYER_NAME', 'OPPONENT', 'SEASON', 'DEF_RATING']].head(10))

NaNs in DEF_RATING: 0
  PLAYER_NAME OPPONENT   SEASON  DEF_RATING
0    AJ Green      MIA  2022-23       112.8
1    AJ Green      CHI  2022-23       111.5
2    AJ Green      MIA  2022-23       112.8
3    AJ Green      ORL  2022-23       113.7
4    AJ Green      IND  2022-23       117.1
5    AJ Green      MEM  2022-23       110.7
6    AJ Green      UTA  2023-24       119.6
7    AJ Green      MIN  2023-24       108.4
8    AJ Green      CHA  2023-24       119.2
9    AJ Green      DEN  2023-24       112.3


In [145]:
# Add opponent pace

from nba_api.stats.endpoints import leaguedashteamstats
import time

# Pull team pace stats for each season
pace_frames = []

for season in ['2020-21', '2021-22', '2022-23', '2023-24', '2024-25']:
    stats = leaguedashteamstats.LeagueDashTeamStats(
        season=season,
        season_type_all_star='Regular Season',
        measure_type_detailed_defense='Advanced'
    )
    time.sleep(1.5)
    df_pace = stats.get_data_frames()[0]
    df_pace['SEASON'] = season
    pace_frames.append(df_pace)

df_pace_all   = pd.concat(pace_frames, ignore_index=True)
df_pace_clean = df_pace_all[['TEAM_NAME', 'SEASON', 'PACE']].copy()

# Protection
if 'PACE' in df_master.columns:
    df_master = df_master.drop(columns=['PACE'])

# Merge
df_master = df_master.merge(
    df_pace_clean[['TEAM_NAME', 'SEASON', 'PACE']],
    left_on=['OPP_TEAM_NAME', 'SEASON'],
    right_on=['TEAM_NAME', 'SEASON'],
    how='left'
)
df_master = df_master.drop(columns=['TEAM_NAME'])

print(f"NaNs in PACE: {df_master['PACE'].isna().sum()}")
print(df_master[['PLAYER_NAME', 'OPPONENT', 'SEASON', 'DEF_RATING', 'PACE']].head(10))

NaNs in PACE: 0
  PLAYER_NAME OPPONENT   SEASON  DEF_RATING    PACE
0    AJ Green      MIA  2022-23       112.8   96.76
1    AJ Green      CHI  2022-23       111.5   99.18
2    AJ Green      MIA  2022-23       112.8   96.76
3    AJ Green      ORL  2022-23       113.7   99.66
4    AJ Green      IND  2022-23       117.1  101.68
5    AJ Green      MEM  2022-23       110.7  101.50
6    AJ Green      UTA  2023-24       119.6  100.26
7    AJ Green      MIN  2023-24       108.4   97.79
8    AJ Green      CHA  2023-24       119.2   97.81
9    AJ Green      DEN  2023-24       112.3   97.43


In [147]:
# Add opponenet stats allowed per game

from nba_api.stats.endpoints import leaguedashteamstats
import time

# Pull per game defensive stats for each season
opp_stats_frames = []

for season in ['2020-21', '2021-22', '2022-23', '2023-24', '2024-25']:
    stats = leaguedashteamstats.LeagueDashTeamStats(
        season=season,
        season_type_all_star='Regular Season',
        measure_type_detailed_defense='Base',
        per_mode_detailed='PerGame'
    )
    time.sleep(1.5)
    df_s = stats.get_data_frames()[0]
    df_s['SEASON'] = season
    opp_stats_frames.append(df_s)

df_opp_stats = pd.concat(opp_stats_frames, ignore_index=True)

# Select and rename
df_opp_stats_clean = df_opp_stats[[
    'TEAM_NAME', 'SEASON',
    'PTS', 'REB', 'AST', 'BLK', 'STL', 'FG3M',
]].copy().rename(columns={
    'PTS':  'OPP_PTS_ALLOWED_PG',
    'REB':  'OPP_REB_ALLOWED_PG',
    'AST':  'OPP_AST_ALLOWED_PG',
    'BLK':  'OPP_BLK_PG',
    'STL':  'OPP_STL_PG',
    'FG3M': 'OPP_3PM_ALLOWED_PG',
})

# Protection
for col in ['OPP_PTS_ALLOWED_PG', 'OPP_REB_ALLOWED_PG', 'OPP_AST_ALLOWED_PG',
            'OPP_BLK_PG', 'OPP_STL_PG', 'OPP_3PM_ALLOWED_PG']:
    if col in df_master.columns:
        df_master = df_master.drop(columns=[col])

# Merge
df_master = df_master.merge(
    df_opp_stats_clean,
    left_on=['OPP_TEAM_NAME', 'SEASON'],
    right_on=['TEAM_NAME', 'SEASON'],
    how='left'
)
df_master = df_master.drop(columns=['TEAM_NAME'])

print(f"NaNs in OPP_PTS_ALLOWED_PG: {df_master['OPP_PTS_ALLOWED_PG'].isna().sum()}")
print(df_master[['PLAYER_NAME', 'OPPONENT', 'SEASON',
                 'OPP_PTS_ALLOWED_PG', 'OPP_REB_ALLOWED_PG']].head(10))

NaNs in OPP_PTS_ALLOWED_PG: 0
  PLAYER_NAME OPPONENT   SEASON  OPP_PTS_ALLOWED_PG  OPP_REB_ALLOWED_PG
0    AJ Green      MIA  2022-23               109.5                40.6
1    AJ Green      CHI  2022-23               113.1                42.4
2    AJ Green      MIA  2022-23               109.5                40.6
3    AJ Green      ORL  2022-23               111.4                43.2
4    AJ Green      IND  2022-23               116.3                41.5
5    AJ Green      MEM  2022-23               116.9                46.6
6    AJ Green      UTA  2023-24               115.7                45.5
7    AJ Green      MIN  2023-24               113.0                43.6
8    AJ Green      CHA  2023-24               106.6                40.3
9    AJ Green      DEN  2023-24               114.9                44.4


In [169]:
# Add usage rate

from nba_api.stats.endpoints import leaguedashplayerstats
import time

# Pull usage stats for each season
usage_frames = []

for season in ['2020-21', '2021-22', '2022-23', '2023-24', '2024-25']:
    stats = leaguedashplayerstats.LeagueDashPlayerStats(
        season=season,
        season_type_all_star='Regular Season',
        measure_type_detailed_defense='Advanced',
        per_mode_detailed='PerGame'
    )
    time.sleep(1.5)
    df_u = stats.get_data_frames()[0]
    df_u['SEASON'] = season
    usage_frames.append(df_u)

df_usage = pd.concat(usage_frames, ignore_index=True)

# Filter to meaningful minutes
df_usage_clean = df_usage[
    df_usage['GP'] >= 10
][['PLAYER_NAME', 'SEASON', 'USG_PCT']].copy()

# Protection
if 'USG_PCT' in df_master.columns:
    df_master = df_master.drop(columns=['USG_PCT'])

# Merge
df_master = df_master.merge(
    df_usage_clean[['PLAYER_NAME', 'SEASON', 'USG_PCT']],
    on=['PLAYER_NAME', 'SEASON'],
    how='left'
)

# Fill any remaining NaNs with league average usage rate
league_avg_usg = df_usage_clean['USG_PCT'].mean()
df_master['USG_PCT'] = df_master['USG_PCT'].fillna(league_avg_usg)

print(f"NaNs in USG_PCT: {df_master['USG_PCT'].isna().sum()}")
print(f"League avg USG used as fallback: {league_avg_usg:.3f}")
print()
print(df_master[['PLAYER_NAME', 'SEASON', 'USG_PCT']].head(10))

NaNs in USG_PCT: 0
League avg USG used as fallback: 0.180

  PLAYER_NAME   SEASON  USG_PCT
0    AJ Green  2022-23    0.159
1    AJ Green  2022-23    0.159
2    AJ Green  2022-23    0.159
3    AJ Green  2022-23    0.159
4    AJ Green  2022-23    0.159
5    AJ Green  2022-23    0.159
6    AJ Green  2023-24    0.150
7    AJ Green  2023-24    0.150
8    AJ Green  2023-24    0.150
9    AJ Green  2023-24    0.150


In [191]:
# Team usage context features
# Relative usage and usage rank within team per season

# Step 1 — Calculate team average usage per season
# Group by team — we need to identify which team each player is on
# We can approximate this using OPP_TEAM_NAME and MATCHUP

# First build a player → team lookup from MATCHUP column
# HOME games: "TEAM vs. OPP" — player's team is left side
# AWAY games: "TEAM @ OPP" — player's team is left side
df_master['PLAYER_TEAM'] = df_master['MATCHUP'].apply(
    lambda x: x.split(' vs.')[0].strip() if 'vs.' in x else x.split(' @')[0].strip()
)

# Verify
print("Sample player team assignments:")
print(df_master[['PLAYER_NAME', 'MATCHUP', 'PLAYER_TEAM']].head(10))
print()
print(f"Unique team codes: {sorted(df_master['PLAYER_TEAM'].unique())}")

Sample player team assignments:
  PLAYER_NAME      MATCHUP PLAYER_TEAM
0    AJ Green    MIL @ MIA         MIL
1    AJ Green    MIL @ CHI         MIL
2    AJ Green  MIL vs. MIA         MIL
3    AJ Green  MIL vs. ORL         MIL
4    AJ Green    MIL @ IND         MIL
5    AJ Green  MIL vs. MEM         MIL
6    AJ Green    MIL @ UTA         MIL
7    AJ Green  MIL vs. MIN         MIL
8    AJ Green  MIL vs. CHA         MIL
9    AJ Green  MIL vs. DEN         MIL

Unique team codes: ['ATL', 'BKN', 'BOS', 'CHA', 'CHI', 'CLE', 'DAL', 'DEN', 'DET', 'GSW', 'HOU', 'IND', 'LAC', 'LAL', 'MEM', 'MIA', 'MIL', 'MIN', 'NOP', 'NYK', 'OKC', 'ORL', 'PHI', 'PHX', 'POR', 'SAC', 'SAS', 'TOR', 'UTA', 'WAS']


In [193]:
# Calculate team average usage per player per season
# This tells us how dominant a player is relative to their teammates

# Calculate average USG_PCT per team per season
team_avg_usg = df_master.groupby(
    ['PLAYER_TEAM', 'SEASON']
)['USG_PCT'].mean().reset_index()

team_avg_usg = team_avg_usg.rename(columns={'USG_PCT': 'TEAM_AVG_USG'})

# Merge team average back into df_master
if 'TEAM_AVG_USG' in df_master.columns:
    df_master = df_master.drop(columns=['TEAM_AVG_USG'])

df_master = df_master.merge(
    team_avg_usg,
    on=['PLAYER_TEAM', 'SEASON'],
    how='left'
)

# Calculate relative usage
# How much more/less does this player use than their average teammate
if 'RELATIVE_USG' in df_master.columns:
    df_master = df_master.drop(columns=['RELATIVE_USG'])

df_master['RELATIVE_USG'] = df_master['USG_PCT'] / df_master['TEAM_AVG_USG']

# Calculate usage rank on team
# 1 = primary option, 2 = second option, etc.
if 'USG_RANK' in df_master.columns:
    df_master = df_master.drop(columns=['USG_RANK'])

# Rank players by usage within each team and season
# We need one USG_PCT per player per season for ranking
player_season_usg = df_master.groupby(
    ['PLAYER_NAME', 'PLAYER_TEAM', 'SEASON']
)['USG_PCT'].first().reset_index()

player_season_usg['USG_RANK'] = player_season_usg.groupby(
    ['PLAYER_TEAM', 'SEASON']
)['USG_PCT'].rank(ascending=False, method='min')

# Merge rank back into df_master
df_master = df_master.merge(
    player_season_usg[['PLAYER_NAME', 'PLAYER_TEAM', 'SEASON', 'USG_RANK']],
    on=['PLAYER_NAME', 'PLAYER_TEAM', 'SEASON'],
    how='left'
)

# Verify
print(f"NaNs in RELATIVE_USG: {df_master['RELATIVE_USG'].isna().sum()}")
print(f"NaNs in USG_RANK:     {df_master['USG_RANK'].isna().sum()}")
print(f"Total columns:        {len(df_master.columns)}")
print()

# Sanity check — top usage players should have rank 1 and high relative usage
print("Sample — star players:")
check_players = ['Luka Dončić', 'Nikola Jokić', 'Victor Wembanyama', 'Stephen Curry']
for player in check_players:
    row = df_master[
        (df_master['PLAYER_NAME'] == player) &
        (df_master['SEASON'] == '2024-25')
    ].iloc[-1]
    print(f"  {player:<25} USG: {row['USG_PCT']:.3f}  "
          f"TeamAvg: {row['TEAM_AVG_USG']:.3f}  "
          f"RelUSG: {row['RELATIVE_USG']:.2f}  "
          f"Rank: {int(row['USG_RANK'])}")

NaNs in RELATIVE_USG: 0
NaNs in USG_RANK:     0
Total columns:        81

Sample — star players:
  Luka Dončić               USG: 0.328  TeamAvg: 0.211  RelUSG: 1.56  Rank: 1
  Nikola Jokić              USG: 0.285  TeamAvg: 0.202  RelUSG: 1.41  Rank: 1
  Victor Wembanyama         USG: 0.300  TeamAvg: 0.199  RelUSG: 1.51  Rank: 1
  Stephen Curry             USG: 0.286  TeamAvg: 0.209  RelUSG: 1.37  Rank: 1


In [195]:
# Check to make sure usage rates look right across players
# Check role players and second options
check_players2 = [
    ('Draymond Green',   '2024-25'),   # facilitator, low usage
    ('Klay Thompson',    '2023-24'),   # second option
    ('Austin Reaves',    '2024-25'),   # role player
    ('Jaylen Brown',     '2024-25'),   # second option on Celtics
]

print("Sample — role players and second options:")
for player, season in check_players2:
    rows = df_master[
        (df_master['PLAYER_NAME'] == player) &
        (df_master['SEASON'] == season)
    ]
    if len(rows) == 0:
        print(f"  {player:<25} not found")
        continue
    row = rows.iloc[-1]
    print(f"  {player:<25} USG: {row['USG_PCT']:.3f}  "
          f"TeamAvg: {row['TEAM_AVG_USG']:.3f}  "
          f"RelUSG: {row['RELATIVE_USG']:.2f}  "
          f"Rank: {int(row['USG_RANK'])}")

Sample — role players and second options:
  Draymond Green            USG: 0.157  TeamAvg: 0.209  RelUSG: 0.75  Rank: 9
  Klay Thompson             USG: 0.234  TeamAvg: 0.207  RelUSG: 1.13  Rank: 3
  Austin Reaves             USG: 0.230  TeamAvg: 0.211  RelUSG: 1.09  Rank: 4
  Jaylen Brown              USG: 0.282  TeamAvg: 0.206  RelUSG: 1.37  Rank: 2


In [201]:
# Addition of Volatility features
# Rolling standard deviation and coefficient of variation
# Captures how "boom or bust" a player is

# Sort by player and date first
df_master = df_master.sort_values(['PLAYER_NAME', 'GAME_DATE']).reset_index(drop=True)

# Protection
for col in ['PTS_std_roll10', 'REB_std_roll10', 
            'PTS_cv_roll10', 'REB_cv_roll10']:
    if col in df_master.columns:
        df_master = df_master.drop(columns=[col])

# Rolling standard deviation over last 10 games
df_master['PTS_std_roll10'] = (
    df_master.groupby('PLAYER_NAME')['PTS']
    .transform(lambda x: x.rolling(window=10, min_periods=3).std().shift(1))
)

df_master['REB_std_roll10'] = (
    df_master.groupby('PLAYER_NAME')['REB']
    .transform(lambda x: x.rolling(window=10, min_periods=3).std().shift(1))
)

# Coefficient of variation = std / mean
# Normalized volatility — accounts for high scorers having naturally higher std
df_master['PTS_cv_roll10'] = (
    df_master['PTS_std_roll10'] / df_master['PTS_roll10'].replace(0, np.nan)
)

df_master['REB_cv_roll10'] = (
    df_master['REB_std_roll10'] / df_master['REB_roll10'].replace(0, np.nan)
)

# Fill NaNs — use season average std as fallback
for col in ['PTS_std_roll10', 'REB_std_roll10']:
    league_avg = df_master[col].mean()
    df_master[col] = df_master[col].fillna(league_avg)
    print(f"League avg {col}: {league_avg:.3f}")

for col in ['PTS_cv_roll10', 'REB_cv_roll10']:
    league_avg = df_master[col].mean()
    df_master[col] = df_master[col].fillna(league_avg)
    print(f"League avg {col}: {league_avg:.3f}")

print()
print(f"NaNs in PTS_std_roll10: {df_master['PTS_std_roll10'].isna().sum()}")
print(f"NaNs in REB_std_roll10: {df_master['REB_std_roll10'].isna().sum()}")
print(f"NaNs in PTS_cv_roll10:  {df_master['PTS_cv_roll10'].isna().sum()}")
print(f"NaNs in REB_cv_roll10:  {df_master['REB_cv_roll10'].isna().sum()}")
print(f"Total columns:          {len(df_master.columns)}")
print()

# Sanity check — volatile players should have high std and cv
print("Volatility check:")
check_players = [
    ('Stephen Curry',   '2024-25'),
    ('Nikola Jokić',    '2024-25'),
    ('Victor Wembanyama', '2024-25'),
    ('Jayson Tatum',    '2024-25'),
]

for player, season in check_players:
    rows = df_master[
        (df_master['PLAYER_NAME'] == player) &
        (df_master['SEASON'] == season)
    ]
    if len(rows) == 0:
        continue
    row = rows.iloc[-1]
    print(f"  {player:<25} "
          f"PTS_std: {row['PTS_std_roll10']:.2f}  "
          f"PTS_cv: {row['PTS_cv_roll10']:.2f}  "
          f"REB_std: {row['REB_std_roll10']:.2f}")

League avg PTS_std_roll10: 6.226
League avg REB_std_roll10: 2.392
League avg PTS_cv_roll10: 0.439
League avg REB_cv_roll10: 0.493

NaNs in PTS_std_roll10: 0
NaNs in REB_std_roll10: 0
NaNs in PTS_cv_roll10:  0
NaNs in REB_cv_roll10:  0
Total columns:          85

Volatility check:
  Stephen Curry             PTS_std: 14.31  PTS_cv: 0.44  REB_std: 3.05
  Nikola Jokić              PTS_std: 9.83  PTS_cv: 0.37  REB_std: 4.83
  Victor Wembanyama         PTS_std: 5.33  PTS_cv: 0.22  REB_std: 1.84
  Jayson Tatum              PTS_std: 6.44  PTS_cv: 0.44  REB_std: 3.28


In [157]:
# Add position mapping to add opponenet vs position feature

# Manual position mapping — G=Guard, F=Forward, C=Center
position_map = {
    'AJ Green': 'G', 'AJ Johnson': 'F', 'Aaron Gordon': 'F',
    'Aaron Nesmith': 'F', 'Aaron Wiggins': 'G', 'Al Horford': 'C',
    'Alex Sarr': 'C', 'Alperen Sengun': 'C', 'Amen Thompson': 'F',
    'Amir Coffey': 'G', 'Andrew Nembhard': 'G', 'Andrew Wiggins': 'F',
    'Anfernee Simons': 'G', 'Anthony Black': 'G', 'Anthony Davis': 'C',
    'Anthony Edwards': 'G', 'Ausar Thompson': 'F', 'Austin Reaves': 'G',
    'Ayo Dosunmu': 'G', 'Bam Adebayo': 'C', 'Ben Simmons': 'F',
    'Bennedict Mathurin': 'G', 'Bilal Coulibaly': 'F', 'Bobby Portis': 'F',
    'Bogdan Bogdanović': 'G', 'Bradley Beal': 'G', 'Brandin Podziemski': 'G',
    'Brandon Boston': 'G', 'Brandon Ingram': 'F', 'Brandon Miller': 'F',
    'Brice Sensabaugh': 'F', 'Brook Lopez': 'C', 'Bruce Brown': 'G',
    'Bub Carrington': 'G', 'Buddy Hield': 'G', 'CJ McCollum': 'G',
    'Cade Cunningham': 'G', 'Caleb Martin': 'F', 'Cam Thomas': 'G',
    'Cameron Johnson': 'F', 'Caris LeVert': 'G', 'Cason Wallace': 'G',
    'Chet Holmgren': 'C', 'Chris Paul': 'G', 'Christian Braun': 'G',
    'Chuma Okeke': 'F', 'Clint Capela': 'C', 'Coby White': 'G',
    'Cody Martin': 'F', 'Cody Williams': 'G', 'Collin Sexton': 'G',
    'Corey Kispert': 'F', 'D\'Angelo Russell': 'G', 'DaQuan Jeffries': 'F',
    'Damian Lillard': 'G', 'Damion Baugh': 'G', 'Daniel Gafford': 'C',
    'Darius Garland': 'G', 'Davion Mitchell': 'G', 'De\'Aaron Fox': 'G',
    'De\'Andre Hunter': 'F', 'De\'Anthony Melton': 'G', 'DeMar DeRozan': 'F',
    'Dean Wade': 'F', 'Deandre Ayton': 'C', 'Dejounte Murray': 'G',
    'Deni Avdija': 'F', 'Dennis Schröder': 'G', 'Dereck Lively II': 'C',
    'Derrick Jones Jr.': 'F', 'Derrick White': 'G', 'Desmond Bane': 'G',
    'Devin Booker': 'G', 'Devin Vassell': 'G', 'Dillon Brooks': 'F',
    'Domantas Sabonis': 'C', 'Donovan Mitchell': 'G', 'Donte DiVincenzo': 'G',
    'Dorian Finney-Smith': 'F', 'Draymond Green': 'F', 'Drew Timme': 'C',
    'Duncan Robinson': 'F', 'Dyson Daniels': 'G', 'Elfrid Payton': 'G',
    'Evan Mobley': 'C', 'Franz Wagner': 'F', 'Fred VanVleet': 'G',
    'Gabe Vincent': 'G', 'Gary Trent Jr.': 'G', 'Georges Niang': 'F',
    'Giannis Antetokounmpo': 'F', 'Goga Bitadze': 'C', 'Gradey Dick': 'G',
    'Grant Williams': 'F', 'Grayson Allen': 'G', 'Guerschon Yabusele': 'F',
    'Harrison Barnes': 'F', 'Haywood Highsmith': 'F', 'Herbert Jones': 'F',
    'Immanuel Quickley': 'G', 'Isaiah Collier': 'G', 'Isaiah Hartenstein': 'C',
    'Isaiah Joe': 'G', 'Ivica Zubac': 'C', 'Ja Morant': 'G',
    'Ja\'Kobe Walter': 'G', 'Jabari Smith Jr.': 'F', 'Jaden Ivey': 'G',
    'Jaden McDaniels': 'F', 'Jaime Jaquez Jr.': 'F', 'Jake LaRavia': 'F',
    'Jakob Poeltl': 'C', 'Jalen Brunson': 'G', 'Jalen Duren': 'C',
    'Jalen Green': 'G', 'Jalen Hood-Schifino': 'G', 'Jalen Johnson': 'F',
    'Jalen Suggs': 'G', 'Jalen Williams': 'G', 'Jalen Wilson': 'F',
    'Jamal Murray': 'G', 'James Harden': 'G', 'Jared McCain': 'G',
    'Jaren Jackson Jr.': 'F', 'Jarrett Allen': 'C', 'Jaylen Brown': 'F',
    'Jaylen Nowell': 'G', 'Jaylen Wells': 'G', 'Jayson Tatum': 'F',
    'Jerami Grant': 'F', 'Jeremy Sochan': 'F', 'Jimmy Butler III': 'F',
    'Joel Embiid': 'C', 'John Collins': 'F', 'Jonathan Kuminga': 'F',
    'Jonathan Mogbo': 'F', 'Jordan Clarkson': 'G', 'Jordan Hawkins': 'G',
    'Jordan Poole': 'G', 'Jose Alvarado': 'G', 'Josh Giddey': 'G',
    'Josh Green': 'G', 'Josh Hart': 'F', 'Jrue Holiday': 'G',
    'Julian Champagnie': 'F', 'Julian Strawther': 'G', 'Julius Randle': 'F',
    'Justin Champagnie': 'F', 'Justin Edwards': 'F', 'Jusuf Nurkić': 'C',
    'KJ Martin': 'F', 'KJ Simpson': 'G', 'Karl-Anthony Towns': 'C',
    'Kawhi Leonard': 'F', 'Keegan Murray': 'F', 'Keion Brooks Jr.': 'F',
    'Kel\'el Ware': 'C', 'Keldon Johnson': 'F', 'Kelly Olynyk': 'C',
    'Kelly Oubre Jr.': 'F', 'Kentavious Caldwell-Pope': 'G', 'Keon Ellis': 'G',
    'Keon Johnson': 'G', 'Kevin Durant': 'F', 'Kevin Huerter': 'G',
    'Keyonte George': 'G', 'Khris Middleton': 'F', 'Killian Hayes': 'G',
    'Klay Thompson': 'G', 'Kris Dunn': 'G', 'Kristaps Porziņģis': 'C',
    'Kyle Filipowski': 'C', 'Kyle Kuzma': 'F', 'Kyrie Irving': 'G',
    'Kyshawn George': 'G', 'LaMelo Ball': 'G', 'Lauri Markkanen': 'F',
    'LeBron James': 'F', 'Lonnie Walker IV': 'G', 'Lonzo Ball': 'G',
    'Luguentz Dort': 'G', 'Luka Dončić': 'G', 'Luke Kennard': 'G',
    'Malcolm Brogdon': 'G', 'Malik Beasley': 'G', 'Malik Monk': 'G',
    'Marcus Bagley': 'F', 'Marcus Smart': 'G', 'Mark Williams': 'C',
    'Matisse Thybulle': 'G', 'Max Christie': 'G', 'Max Strus': 'G',
    'Michael Porter Jr.': 'F', 'Mikal Bridges': 'F', 'Mike Conley': 'G',
    'Miles Bridges': 'F', 'Miles McBride': 'G', 'Moses Moody': 'G',
    'Myles Turner': 'C', 'Naji Marshall': 'F', 'Naz Reid': 'C',
    'Nic Claxton': 'C', 'Nick Richards': 'C', 'Nick Smith Jr.': 'G',
    'Nickeil Alexander-Walker': 'G', 'Nikola Jokić': 'C', 'Nikola Jović': 'F',
    'Nikola Vučević': 'C', 'Noah Clowney': 'F', 'Norman Powell': 'G',
    'OG Anunoby': 'F', 'Ochai Agbaji': 'G', 'Onyeka Okongwu': 'C',
    'Oshae Brissett': 'F', 'P.J. Washington': 'F', 'Paolo Banchero': 'F',
    'Pascal Siakam': 'F', 'Patrick Williams': 'F', 'Paul George': 'F',
    'Payton Pritchard': 'G', 'Peyton Watson': 'F', 'Precious Achiuwa': 'F',
    'Quentin Grimes': 'G', 'RJ Barrett': 'F', 'Royce O\'Neale': 'F',
    'Rudy Gobert': 'C', 'Rui Hachimura': 'F', 'Russell Westbrook': 'G',
    'Sam Hauser': 'F', 'Santi Aldama': 'F', 'Scoot Henderson': 'G',
    'Scottie Barnes': 'F', 'Scotty Pippen Jr.': 'G', 'Shaedon Sharpe': 'G',
    'Shai Gilgeous-Alexander': 'G', 'Spencer Dinwiddie': 'G',
    'Stephen Curry': 'G', 'Stephon Castle': 'G', 'Svi Mykhailiuk': 'G',
    'Tari Eason': 'F', 'Taurean Prince': 'F', 'Taylor Hendricks': 'F',
    'Terance Mann': 'F', 'Terry Rozier': 'G', 'Tidjane Salaün': 'F',
    'Tim Hardaway Jr.': 'G', 'Tobias Harris': 'F', 'Tolu Smith': 'C',
    'Tosan Evbuomwan': 'F', 'Toumani Camara': 'F', 'Trae Young': 'G',
    'Tre Mann': 'G', 'Trendon Watford': 'F', 'Trey Murphy III': 'F',
    'Tristan da Silva': 'F', 'Tyler Herro': 'G', 'Tyrese Haliburton': 'G',
    'Tyrese Martin': 'G', 'Tyrese Maxey': 'G', 'Tyson Etienne': 'G',
    'Tyus Jones': 'G', 'Victor Wembanyama': 'C', 'Vít Krejčí': 'G',
    'Walker Kessler': 'C', 'Wendell Carter Jr.': 'C', 'Yves Missi': 'C',
    'Zaccharie Risacher': 'F', 'Zach Edey': 'C', 'Zach LaVine': 'G',
    'Ziaire Williams': 'F', 'Zion Williamson': 'F',
}

df_master['POSITION'] = df_master['PLAYER_NAME'].map(position_map)

mapped = df_master['POSITION'].notna().sum()
total  = len(df_master)
print(f"Players mapped: {mapped:,} / {total:,} ({mapped/total*100:.1f}%)")
print()
unmapped = df_master[df_master['POSITION'].isna()]['PLAYER_NAME'].unique()
print(f"Unmapped: {unmapped if len(unmapped) > 0 else 'None ✅'}")

Players mapped: 54,701 / 54,701 (100.0%)

Unmapped: None ✅


In [159]:
# Add opponent vs position stats

# Calculate average stats allowed by each team to each position per season
df_pos_stats = df_master[['OPPONENT', 'SEASON', 'POSITION',
                           'PTS', 'REB', 'AST', 'BLK', 'STL', 'FG3M']].copy()

pos_def = df_pos_stats.groupby(['OPPONENT', 'SEASON', 'POSITION'])[
    ['PTS', 'REB', 'AST', 'BLK', 'STL', 'FG3M']
].mean().round(2).reset_index()

pos_def = pos_def.rename(columns={
    'PTS':  'OPP_PTS_VS_POS', 'REB':  'OPP_REB_VS_POS',
    'AST':  'OPP_AST_VS_POS', 'BLK':  'OPP_BLK_VS_POS',
    'STL':  'OPP_STL_VS_POS', 'FG3M': 'OPP_3PM_VS_POS',
})

# Protection
for col in ['OPP_PTS_VS_POS', 'OPP_REB_VS_POS', 'OPP_AST_VS_POS',
            'OPP_BLK_VS_POS', 'OPP_STL_VS_POS', 'OPP_3PM_VS_POS']:
    if col in df_master.columns:
        df_master = df_master.drop(columns=[col])

df_master = df_master.merge(
    pos_def,
    on=['OPPONENT', 'SEASON', 'POSITION'],
    how='left'
)

print(f"NaNs in OPP_PTS_VS_POS: {df_master['OPP_PTS_VS_POS'].isna().sum()}")
print(f"Total columns: {len(df_master.columns)}")
print(df_master[['PLAYER_NAME', 'POSITION', 'OPPONENT', 'SEASON',
                 'OPP_PTS_VS_POS', 'OPP_REB_VS_POS']].head(10))

NaNs in OPP_PTS_VS_POS: 0
Total columns: 69
  PLAYER_NAME POSITION OPPONENT   SEASON  OPP_PTS_VS_POS  OPP_REB_VS_POS
0    AJ Green        G      MIA  2022-23           16.19            3.79
1    AJ Green        G      CHI  2022-23           16.64            4.00
2    AJ Green        G      MIA  2022-23           16.19            3.79
3    AJ Green        G      ORL  2022-23           15.74            3.65
4    AJ Green        G      IND  2022-23           17.51            4.04
5    AJ Green        G      MEM  2022-23           15.66            3.99
6    AJ Green        G      UTA  2023-24           16.73            3.98
7    AJ Green        G      MIN  2023-24           14.82            3.46
8    AJ Green        G      CHA  2023-24           14.50            3.94
9    AJ Green        G      DEN  2023-24           15.03            3.68


In [161]:
# Build rolling team-vs-position features

# Define merge_cols first — needed for per-player conversion step
merge_cols = {
    'ALLOWED_PTS_roll5':  'OPP_PTS_VS_POS_roll5',
    'ALLOWED_REB_roll5':  'OPP_REB_VS_POS_roll5',
    'ALLOWED_AST_roll5':  'OPP_AST_VS_POS_roll5',
    'ALLOWED_BLK_roll5':  'OPP_BLK_VS_POS_roll5',
    'ALLOWED_STL_roll5':  'OPP_STL_VS_POS_roll5',
    'ALLOWED_FG3M_roll5': 'OPP_3PM_VS_POS_roll5',
}

# Build game-by-game team vs position stats
stats_to_roll = ['PTS', 'REB', 'AST', 'BLK', 'STL', 'FG3M']

team_vs_pos_daily = df_master.groupby(
    ['OPPONENT', 'GAME_DATE', 'POSITION']
)[stats_to_roll].sum().reset_index()

team_vs_pos_daily = team_vs_pos_daily.rename(columns={
    'PTS': 'ALLOWED_PTS', 'REB': 'ALLOWED_REB', 'AST': 'ALLOWED_AST',
    'BLK': 'ALLOWED_BLK', 'STL': 'ALLOWED_STL', 'FG3M': 'ALLOWED_FG3M',
})

# Sort for rolling
team_vs_pos_daily = team_vs_pos_daily.sort_values(
    ['OPPONENT', 'POSITION', 'GAME_DATE']
).reset_index(drop=True)

# Calculate 5-game rolling averages
allowed_stats = ['ALLOWED_PTS', 'ALLOWED_REB', 'ALLOWED_AST',
                 'ALLOWED_BLK', 'ALLOWED_STL', 'ALLOWED_FG3M']

for stat in allowed_stats:
    roll_col = f'{stat}_roll5'
    team_vs_pos_daily[roll_col] = (
        team_vs_pos_daily.groupby(['OPPONENT', 'POSITION'])[stat]
        .transform(lambda x: x.rolling(window=5, min_periods=1).mean().shift(1))
    )

# Convert rolling totals to per-player averages
position_counts = df_master.groupby(
    ['OPPONENT', 'GAME_DATE', 'POSITION']
).size().reset_index(name='player_count')

position_counts = position_counts.sort_values(['OPPONENT', 'POSITION', 'GAME_DATE'])
position_counts['player_count_roll5'] = (
    position_counts.groupby(['OPPONENT', 'POSITION'])['player_count']
    .transform(lambda x: x.rolling(window=5, min_periods=1).mean().shift(1))
)

team_vs_pos_daily = team_vs_pos_daily.merge(
    position_counts[['OPPONENT', 'GAME_DATE', 'POSITION', 'player_count_roll5']],
    on=['OPPONENT', 'GAME_DATE', 'POSITION'],
    how='left'
)

for col in merge_cols.keys():
    team_vs_pos_daily[col] = (
        team_vs_pos_daily[col] / team_vs_pos_daily['player_count_roll5']
    )

# Merge into df_master
team_vs_pos_merge = team_vs_pos_daily[
    ['OPPONENT', 'GAME_DATE', 'POSITION'] + list(merge_cols.keys())
].rename(columns=merge_cols)

for col in merge_cols.values():
    if col in df_master.columns:
        df_master = df_master.drop(columns=[col])

df_master = df_master.merge(
    team_vs_pos_merge,
    on=['OPPONENT', 'GAME_DATE', 'POSITION'],
    how='left'
)

# Fill NaNs with static season average as fallback
roll_cols   = list(merge_cols.values())
static_cols = ['OPP_PTS_VS_POS', 'OPP_REB_VS_POS', 'OPP_AST_VS_POS',
               'OPP_BLK_VS_POS', 'OPP_STL_VS_POS', 'OPP_3PM_VS_POS']

for roll_col, static_col in zip(roll_cols, static_cols):
    df_master[roll_col] = df_master[roll_col].fillna(df_master[static_col])

print(f"NaNs in OPP_PTS_VS_POS_roll5: {df_master['OPP_PTS_VS_POS_roll5'].isna().sum()}")
print(f"Total columns: {len(df_master.columns)}")

NaNs in OPP_PTS_VS_POS_roll5: 0
Total columns: 75


In [163]:
# Add rolling usage rates

# Sort by player and date
df_master = df_master.sort_values(['PLAYER_NAME', 'GAME_DATE']).reset_index(drop=True)

# Protection
if 'USG_PCT_roll5' in df_master.columns:
    df_master = df_master.drop(columns=['USG_PCT_roll5'])

# Rolling usage rate per player
df_master['USG_PCT_roll5'] = (
    df_master.groupby('PLAYER_NAME')['USG_PCT']
    .transform(lambda x: x.rolling(window=5, min_periods=1).mean().shift(1))
)

# Fill NaNs with static USG_PCT
df_master['USG_PCT_roll5'] = df_master['USG_PCT_roll5'].fillna(df_master['USG_PCT'])

print(f"NaNs in USG_PCT_roll5: {df_master['USG_PCT_roll5'].isna().sum()}")
print(f"Total columns: {len(df_master.columns)}")
print()
print("Usage rate comparison:")
for player in ['Victor Wembanyama', 'Nikola Jokić', 'Stephen Curry', 'Jayson Tatum']:
    latest = df_master[df_master['PLAYER_NAME'] == player].iloc[-1]
    print(f"  {player:<25} USG_PCT: {latest['USG_PCT']:.3f}  USG_PCT_roll5: {latest['USG_PCT_roll5']:.3f}")

NaNs in USG_PCT_roll5: 572
Total columns: 76

Usage rate comparison:
  Victor Wembanyama         USG_PCT: 0.300  USG_PCT_roll5: 0.300
  Nikola Jokić              USG_PCT: 0.285  USG_PCT_roll5: 0.285
  Stephen Curry             USG_PCT: 0.286  USG_PCT_roll5: 0.286
  Jayson Tatum              USG_PCT: 0.301  USG_PCT_roll5: 0.301


In [165]:
# Pull playoff game logs

from nba_api.stats.endpoints import playergamelog
import time

SEASONS = ['2020-21', '2021-22', '2022-23', '2023-24', '2024-25']
SLEEP   = 1.5

playoff_data = []
total        = len(player_pull_list) * len(SEASONS)
current      = 0

for _, player in player_pull_list.iterrows():
    for season in SEASONS:
        current += 1
        try:
            log = playergamelog.PlayerGameLog(
                player_id=player['PLAYER_ID'],
                season=season,
                season_type_all_star='Playoffs'
            )
            time.sleep(SLEEP)

            df_raw = log.get_data_frames()[0]

            if len(df_raw) == 0:
                continue

            df_cleaned = clean_player_log(df_raw, player['PLAYER_NAME'], season)

            if len(df_cleaned) == 0:
                continue

            playoff_data.append(df_cleaned)
            print(f"[{current}/{total}] ✅ {player['PLAYER_NAME']} {season} — {len(df_cleaned)} rows")

        except Exception as e:
            print(f"[{current}/{total}] ❌ {player['PLAYER_NAME']} {season} — {e}")
            time.sleep(SLEEP)
            continue

df_playoffs = pd.concat(playoff_data, ignore_index=True)

print()
print(f"═══════════════════════════════════")
print(f"  Playoff rows:   {len(df_playoffs):,}")
print(f"  Players:        {df_playoffs['PLAYER_NAME'].nunique()}")
print(f"  Seasons:        {sorted(df_playoffs['SEASON'].unique())}")
print(f"═══════════════════════════════════")

[11/1355] ✅ Aaron Gordon 2020-21 — 5 rows
[13/1355] ✅ Aaron Gordon 2022-23 — 15 rows
[14/1355] ✅ Aaron Gordon 2023-24 — 7 rows
[15/1355] ✅ Aaron Gordon 2024-25 — 9 rows
[19/1355] ✅ Aaron Nesmith 2023-24 — 12 rows
[20/1355] ✅ Aaron Nesmith 2024-25 — 18 rows
[24/1355] ✅ Aaron Wiggins 2023-24 — 1 rows
[25/1355] ✅ Aaron Wiggins 2024-25 — 4 rows
[27/1355] ✅ Al Horford 2021-22 — 18 rows
[28/1355] ✅ Al Horford 2022-23 — 15 rows
[29/1355] ✅ Al Horford 2023-24 — 14 rows
[30/1355] ✅ Al Horford 2024-25 — 6 rows
[40/1355] ✅ Alperen Sengun 2024-25 — 2 rows
[45/1355] ✅ Amen Thompson 2024-25 — 2 rows
[54/1355] ✅ Andrew Nembhard 2023-24 — 12 rows
[55/1355] ✅ Andrew Nembhard 2024-25 — 18 rows
[57/1355] ✅ Andrew Wiggins 2021-22 — 17 rows
[58/1355] ✅ Andrew Wiggins 2022-23 — 8 rows
[73/1355] ✅ Anthony Davis 2022-23 — 11 rows
[77/1355] ✅ Anthony Edwards 2021-22 — 1 rows
[79/1355] ✅ Anthony Edwards 2023-24 — 11 rows
[80/1355] ✅ Anthony Edwards 2024-25 — 10 rows
[85/1355] ✅ Ausar Thompson 2024-25 — 1 rows
[

In [171]:
# Merge playoffs into df_master

print(f"Regular season rows: {len(df_master):,}")
print(f"Playoff rows to add: {len(df_playoffs):,}")
print()

df_master = pd.concat([df_master, df_playoffs], ignore_index=True)
df_master = df_master.sort_values(
    ['PLAYER_NAME', 'GAME_DATE']
).reset_index(drop=True)

print(f"Total rows after merge: {len(df_master):,}")
print()
print("Rows per season:")
print(df_master.groupby('SEASON').size())

Regular season rows: 54,701
Playoff rows to add: 2,284

Total rows after merge: 56,985

Rows per season:
SEASON
2020-21     7675
2021-22     9965
2022-23    11737
2023-24    13240
2024-25    14368
dtype: int64


In [173]:
# Fill missing columns for playoff rows

# Fill POSITION for playoff rows
df_master['POSITION'] = df_master['POSITION'].fillna(
    df_master['PLAYER_NAME'].map(position_map)
)

# Fill OPPONENT and OPP_TEAM_NAME for playoff rows
df_master['OPPONENT'] = df_master['OPPONENT'].fillna(
    df_master['MATCHUP'].apply(lambda x: x.split()[-1])
)
df_master['OPP_TEAM_NAME'] = df_master['OPP_TEAM_NAME'].fillna(
    df_master['OPPONENT'].map(team_name_map)
)

print(f"NaNs in POSITION:      {df_master['POSITION'].isna().sum()}")
print(f"NaNs in OPPONENT:      {df_master['OPPONENT'].isna().sum()}")
print(f"NaNs in OPP_TEAM_NAME: {df_master['OPP_TEAM_NAME'].isna().sum()}")
print(f"Total rows:            {len(df_master):,}")

NaNs in POSITION:      0
NaNs in OPPONENT:      0
NaNs in OPP_TEAM_NAME: 0
Total rows:            56,985


In [175]:
# Add playoff flag

def is_playoff_game(row):
    date   = row['GAME_DATE']
    season = row['SEASON']

    playoff_starts = {
        '2020-21': '2021-05-22',
        '2021-22': '2022-04-16',
        '2022-23': '2023-04-15',
        '2023-24': '2024-04-20',
        '2024-25': '2025-04-19',
    }

    if season in playoff_starts:
        return 1 if date >= pd.Timestamp(playoff_starts[season]) else 0
    return 0

# Protection
if 'IS_PLAYOFF' in df_master.columns:
    df_master = df_master.drop(columns=['IS_PLAYOFF'])

df_master['IS_PLAYOFF'] = df_master.apply(is_playoff_game, axis=1)

print("Playoff vs Regular season games:")
print(df_master['IS_PLAYOFF'].value_counts())
print()
print(f"Playoff games:        {df_master['IS_PLAYOFF'].sum():,}")
print(f"Regular season games: {(df_master['IS_PLAYOFF']==0).sum():,}")
print()
print("Sample playoff games:")
print(df_master[df_master['IS_PLAYOFF'] == 1][
    ['PLAYER_NAME', 'GAME_DATE', 'SEASON', 'IS_PLAYOFF']
].head(10))

Playoff vs Regular season games:
IS_PLAYOFF
0    54701
1     2284
Name: count, dtype: int64

Playoff games:        2,284
Regular season games: 54,701

Sample playoff games:
      PLAYER_NAME  GAME_DATE   SEASON  IS_PLAYOFF
136  Aaron Gordon 2021-06-03  2020-21           1
137  Aaron Gordon 2021-06-07  2020-21           1
138  Aaron Gordon 2021-06-09  2020-21           1
139  Aaron Gordon 2021-06-11  2020-21           1
140  Aaron Gordon 2021-06-13  2020-21           1
273  Aaron Gordon 2023-04-29  2022-23           1
274  Aaron Gordon 2023-05-01  2022-23           1
275  Aaron Gordon 2023-05-05  2022-23           1
276  Aaron Gordon 2023-05-07  2022-23           1
277  Aaron Gordon 2023-05-09  2022-23           1


In [185]:
# Fix NaNs in USG_PCT_roll5 for playoff rows
# Build a lookup of player season usage from regular season rows
usg_lookup = df_master[
    df_master['IS_PLAYOFF'] == 0
].groupby(['PLAYER_NAME', 'SEASON'])['USG_PCT'].first().reset_index()

# Fill missing USG_PCT for playoff rows using regular season value
df_master = df_master.merge(
    usg_lookup.rename(columns={'USG_PCT': 'USG_PCT_fill'}),
    on=['PLAYER_NAME', 'SEASON'],
    how='left'
)

df_master['USG_PCT'] = df_master['USG_PCT'].fillna(df_master['USG_PCT_fill'])
df_master = df_master.drop(columns=['USG_PCT_fill'])

# Now recalculate rolling usage
df_master = df_master.sort_values(['PLAYER_NAME', 'GAME_DATE']).reset_index(drop=True)

df_master['USG_PCT_roll5'] = (
    df_master.groupby('PLAYER_NAME')['USG_PCT']
    .transform(lambda x: x.rolling(window=5, min_periods=1).mean().shift(1))
)

# Fill any remaining NaNs
df_master['USG_PCT_roll5'] = df_master['USG_PCT_roll5'].fillna(df_master['USG_PCT'])

print(f"NaNs in USG_PCT:       {df_master['USG_PCT'].isna().sum()}")
print(f"NaNs in USG_PCT_roll5: {df_master['USG_PCT_roll5'].isna().sum()}")
print()
print("Sample playoff rows:")
print(df_master[df_master['IS_PLAYOFF'] == 1][
    ['PLAYER_NAME', 'SEASON', 'USG_PCT', 'USG_PCT_roll5']
].head(5))

NaNs in USG_PCT:       0
NaNs in USG_PCT_roll5: 0

Sample playoff rows:
      PLAYER_NAME   SEASON  USG_PCT  USG_PCT_roll5
136  Aaron Gordon  2020-21    0.204          0.204
137  Aaron Gordon  2020-21    0.204          0.204
138  Aaron Gordon  2020-21    0.204          0.204
139  Aaron Gordon  2020-21    0.204          0.204
140  Aaron Gordon  2020-21    0.204          0.204


In [203]:
# Save

df_master.to_csv('nba_master_dataset.csv', index=False)

df_verify = pd.read_csv('nba_master_dataset.csv', nrows=1)

print(f"✅ Saved successfully")
print(f"Rows:    {len(df_master):,}")
print(f"Columns: {len(df_verify.columns)}")
print()

✅ Saved successfully
Rows:    56,985
Columns: 85

